In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
folder="/content/drive/MyDrive/資料科學自學聖經/ch12"

## 12.2 查詢歷史股票資料

安裝 twstock 模組命令

In [3]:
!pip install twstock

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.1/626.1 kB 12.4 MB/s eta 0:00:00


先匯入模組程式庫

In [4]:
import twstock

利用Stock('股票代號')方法查詢個股歷史股票資料

In [5]:
stock = twstock.Stock('2330')

利用Stock物件的屬性讀取指定的歷史資料 (※預設讀取近31日的歷史紀錄)

In [7]:
print(stock.price)

[903.0, 815.0, 880.0, 920.0, 896.0, 934.0, 940.0, 941.0, 948.0, 943.0, 969.0, 973.0, 973.0, 958.0, 951.0, 949.0, 950.0, 942.0, 964.0, 943.0, 944.0, 948.0, 940.0, 889.0, 902.0, 918.0, 899.0, 904.0, 901.0, 940.0, 947.0]


In [8]:
print("日期：",stock.date[-1])
print("開盤價：",stock.open[-1])
print("最高價：",stock.high[-1])
print("最低價：",stock.low[-1])
print("收盤價：",stock.price[-1])

日期： 2024-09-13 00:00:00
開盤價： 955.0
最高價： 955.0
最低價： 939.0
收盤價： 947.0


利用Stock物件的fetch()方法,讀取指定期間的歷史資料

In [ ]:
stock.fetch(2024,7)

[Data(date=datetime.datetime(2024, 7, 1, 0, 0), capacity=20936005, turnover=20320957284, open=968.0, high=977.0, low=965.0, close=968.0, change=2.0, transaction=38293),
 Data(date=datetime.datetime(2024, 7, 2, 0, 0), capacity=27992930, turnover=26971516491, open=967.0, high=971.0, low=959.0, close=960.0, change=-8.0, transaction=38928),
 Data(date=datetime.datetime(2024, 7, 3, 0, 0), capacity=25022531, turnover=24386705873, open=976.0, high=979.0, low=967.0, close=979.0, change=19.0, transaction=36096),
 Data(date=datetime.datetime(2024, 7, 4, 0, 0), capacity=47251502, turnover=47347126144, open=1000.0, high=1010.0, low=997.0, close=1005.0, change=26.0, transaction=92002),
 Data(date=datetime.datetime(2024, 7, 5, 0, 0), capacity=21735614, turnover=21827958195, open=1005.0, high=1010.0, low=1000.0, close=1005.0, change=0.0, transaction=35802),
 Data(date=datetime.datetime(2024, 7, 8, 0, 0), capacity=45678332, turnover=47210175550, open=1005.0, high=1050.0, low=1000.0, close=1035.0, chan

In [ ]:
stock.fetch_from(2022,12)

[Data(date=datetime.datetime(2022, 12, 1, 0, 0), capacity=43684491, turnover=21976936461, open=506.0, high=508.0, low=498.5, close=498.5, change=8.5, transaction=46181),
 Data(date=datetime.datetime(2022, 12, 2, 0, 0), capacity=29909021, turnover=14745810388, open=490.0, high=497.0, low=490.0, close=492.5, change=-6.0, transaction=31799),
 Data(date=datetime.datetime(2022, 12, 5, 0, 0), capacity=33048153, turnover=16272219716, open=491.5, high=497.5, low=489.0, close=489.0, change=-3.5, transaction=31739),
 Data(date=datetime.datetime(2022, 12, 6, 0, 0), capacity=44964237, turnover=21703863181, open=488.0, high=489.0, low=478.0, close=478.0, change=-11.0, transaction=44448),
 Data(date=datetime.datetime(2022, 12, 7, 0, 0), capacity=39978821, turnover=19128832587, open=477.0, high=485.5, low=475.0, close=475.0, change=-3.0, transaction=43603),
 Data(date=datetime.datetime(2022, 12, 8, 0, 0), capacity=28348573, turnover=13366736635, open=475.0, high=475.0, low=467.0, close=471.5, change=

## 12.2.2 下載全年個股資料

In [ ]:
!pip install twstock

In [ ]:
import csv, os, time
import twstock

datayear = 2024
startmonth = 7 #可改為4,7,10
filepath = 'twstock' + str(datayear) + '.csv' #在根目錄產生twstockXXXX.csv檔案
title=["日期","成交股數","成交金額","開盤價",
       "最高價","最低價","收盤價","漲跌價差",
       "成交筆數"]
for i in range(startmonth, 13):
    outputfile = open(filepath, 'a', newline='', encoding='big5') # a:表示寫入資料時是將資料附加到檔案後面。
    #開啟檔案(※為避免被鎖IP,故機此行程式放入迴圈內,以利放在記憶緩衝區的資料寫入CSV檔案中)
    outputwriter = csv.writer(outputfile)
    stock = twstock.Stock('2330')
    stocklist=stock.fetch(datayear,i)
    data=[]
    for stock in stocklist:
        strdate=stock.date.strftime("%Y-%m-%d")
        li=[strdate,stock.capacity,stock.turnover,
            stock.open,stock.high,stock.low,stock.close,
            stock.change,stock.transaction]
        data.append(li)
    if i==1:
        outputwriter.writerow(title)
    for dataline in (data):
        outputwriter.writerow(dataline)
    time.sleep(2) #延遲2秒,避免檔案來不及寫入
    outputfile.close() #關閉檔案

# 12.3 實作台灣股票市場股價預測

## 12.3.1 資料預處理

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense

df = pd.read_csv('/content/drive/MyDrive/資料科學自學聖經/ch12/twstock_all.csv', encoding='big5')

取得收盤價資料

In [ ]:
dfprice = pd.DataFrame(df['收盤價'])
dfprice

,收盤價
0,87.2
1,84.5
2,85.2
3,86.9
4,86.5
...,...
1667,104.0
1668,104.5
1669,105.5
1670,105.0


建立RNN資料串列

In [ ]:
sequence_length = 10 # 特徵值(前10天資料)
data = [] #標籤值(Lable Value)
for i in range(len(dfprice) - sequence_length):
    data.append(dfprice[i: i + sequence_length + 1])
print(data[0])# 第11天資料:標籤值(Lable Value)

     收盤價
0   87.2
1   84.5
2   85.2
3   86.9
4   86.5
5   85.7
6   86.3
7   85.3
8   84.8
9   84.1
10  83.7


資料標準化

In [ ]:
sequence_length = 10
scaler = MinMaxScaler()
dfprice = scaler.fit_transform(dfprice)
data = []
for i in range(len(dfprice) - sequence_length):
    data.append(dfprice[i: i + sequence_length + 1])

分割訓練及測試資料

In [ ]:
reshaped_data = np.array(data)
x = reshaped_data[:, :-1] #x:特徵陣列,取得除了最後一欄以外的所有資料(1-10欄)
y = reshaped_data[:, -1]  #y:標籤陣列,取得最後一欄資料(第11欄)

In [ ]:
split_boundary = int(reshaped_data.shape[0] * 0.8)
train_x = x[: split_boundary]
test_x = x[split_boundary:]
train_y = y[: split_boundary]
test_y = y[split_boundary:]
print('訓練資料數量：{}'.format(len(train_x)))
print('測試資料數量：{}'.format(len(test_x)))

訓練資料數量：1329
測試資料數量：333


## 12.3.2 建立及訓練循環神經網路

建立Sequential()模型與LSTM層

In [ ]:
model = Sequential()
model.add(LSTM(input_shape=(10,1), units=256, unroll=False))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


建立輸出層

In [ ]:
model.add(Dense(units=1))

查看權重數量

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 256)                 │         264,192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │             257 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 264,449 (1.01 MB)

 Trainable params: 264,449 (1.01 MB)

 Non-trainable params: 0 (0.00 B)

訓練與儲存模型

In [ ]:
model.compile(loss="mse", optimizer="adam",
              metrics=['accuracy'])
model.fit(train_x, train_y, batch_size=100,
          epochs=100, validation_split=0.2,
          verbose=2)
model.save('stock_model.h5')

Epoch 1/100
11/11 - 4s - 320ms/step - accuracy: 0.0000e+00 - loss: 0.0309 - val_accuracy: 0.0038 - val_loss: 0.0023
Epoch 2/100
11/11 - 0s - 16ms/step - accuracy: 0.0000e+00 - loss: 0.0058 - val_accuracy: 0.0038 - val_loss: 0.0046
Epoch 3/100
11/11 - 0s - 11ms/step - accuracy: 0.0000e+00 - loss: 0.0029 - val_accuracy: 0.0038 - val_loss: 0.0018
Epoch 4/100
11/11 - 0s - 12ms/step - accuracy: 0.0000e+00 - loss: 0.0020 - val_accuracy: 0.0038 - val_loss: 0.0020
Epoch 5/100
11/11 - 0s - 8ms/step - accuracy: 0.0000e+00 - loss: 0.0015 - val_accuracy: 0.0038 - val_loss: 0.0017
Epoch 6/100
11/11 - 0s - 8ms/step - accuracy: 0.0000e+00 - loss: 0.0015 - val_accuracy: 0.0038 - val_loss: 0.0016
Epoch 7/100
11/11 - 0s - 13ms/step - accuracy: 0.0000e+00 - loss: 0.0015 - val_accuracy: 0.0038 - val_loss: 0.0016
Epoch 8/100
11/11 - 0s - 9ms/step - accuracy: 0.0000e+00 - loss: 0.0014 - val_accuracy: 0.0038 - val_loss: 0.0016
Epoch 9/100
11/11 - 0s - 8ms/step - accuracy: 0.0000e+00 - loss: 0.0014 - val_accu

## 12.3.3 完整股價預測程式碼

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense

df = pd.read_csv('/content/drive/MyDrive/資料科學自學聖經/ch12/twstock_all.csv', encoding='big5')
dfprice = pd.DataFrame(df['收盤價'])
sequence_length = 10
scaler = MinMaxScaler()
dfprice = scaler.fit_transform(dfprice)
data = []
for i in range(len(dfprice) - sequence_length):
    data.append(dfprice[i: i + sequence_length + 1])
reshaped_data = np.array(data)
x = reshaped_data[:, :-1]
y = reshaped_data[:, -1]
split_boundary = int(reshaped_data.shape[0] * 0.8)
train_x = x[: split_boundary]
test_x = x[split_boundary:]
train_y = y[: split_boundary]
test_y = y[split_boundary:]

model = Sequential()
model.add(LSTM(input_shape=(10,1), units=256, unroll=False))
model.add(Dense(units=1))
model.compile(loss="mse", optimizer="adam", metrics=['accuracy'])
model.fit(train_x, train_y, batch_size=100,
          epochs=100, validation_split=0.2, verbose=2)
model.save('stock_model.h5')

Epoch 1/100


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


11/11 - 2s - 175ms/step - accuracy: 0.0000e+00 - loss: 0.0451 - val_accuracy: 0.0038 - val_loss: 0.0027
Epoch 2/100
11/11 - 0s - 26ms/step - accuracy: 0.0000e+00 - loss: 0.0074 - val_accuracy: 0.0038 - val_loss: 0.0031
Epoch 3/100
11/11 - 0s - 10ms/step - accuracy: 0.0000e+00 - loss: 0.0036 - val_accuracy: 0.0038 - val_loss: 0.0019
Epoch 4/100
11/11 - 0s - 13ms/step - accuracy: 0.0000e+00 - loss: 0.0023 - val_accuracy: 0.0038 - val_loss: 0.0025
Epoch 5/100
11/11 - 0s - 11ms/step - accuracy: 0.0000e+00 - loss: 0.0018 - val_accuracy: 0.0038 - val_loss: 0.0017
Epoch 6/100
11/11 - 0s - 9ms/step - accuracy: 0.0000e+00 - loss: 0.0015 - val_accuracy: 0.0038 - val_loss: 0.0017
Epoch 7/100
11/11 - 0s - 12ms/step - accuracy: 0.0000e+00 - loss: 0.0015 - val_accuracy: 0.0038 - val_loss: 0.0017
Epoch 8/100
11/11 - 0s - 11ms/step - accuracy: 0.0000e+00 - loss: 0.0015 - val_accuracy: 0.0038 - val_loss: 0.0016
Epoch 9/100
11/11 - 0s - 14ms/step - accuracy: 0.0000e+00 - loss: 0.0015 - val_accuracy: 0.0

## 12.3.4 預測股票收盤價

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
#from keras.models import load_model
from keras.models import load_model
# from keras.losses import mse  # Import the mse loss function

df = pd.read_csv('/content/drive/MyDrive/資料科學自學聖經/ch12/twstock_all.csv', encoding='big5')
dfprice = pd.DataFrame(df['收盤價'])
sequence_length = 10
scaler = MinMaxScaler()
dfprice = scaler.fit_transform(dfprice)
data = []
for i in range(len(dfprice) - sequence_length):
    data.append(dfprice[i: i + sequence_length + 1])
reshaped_data = np.array(data)
x = reshaped_data[:, :-1]
y = reshaped_data[:, -1]
split_boundary = int(reshaped_data.shape[0] * 0.8)
train_x = x[: split_boundary]
test_x = x[split_boundary:]
train_y = y[: split_boundary]
test_y = y[split_boundary:]

#model = load_model('/content/drive/MyDrive/資料科學自學聖經/ch12/testStock.h5')
model = load_model('/content/stock_model.h5')#自建的模型
predict = model.predict(test_x)
predict = scaler.inverse_transform(predict)
test_y = scaler.inverse_transform(test_y)

plt.figure(figsize=(12,6))
plt.plot(predict, 'b-')
plt.plot(test_y, 'r-')

plt.legend(['predict', 'realdata'])
plt.show()


TypeError: Could not locate function 'mse'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'keras.metrics', 'class_name': 'function', 'config': 'mse', 'registered_name': 'mse'}